# WASM Output Level Analysis — Full Provenance Chain

**Question:** Why do the WAV recordings from Sonic Pi Web sound louder than desktop Sonic Pi for the same code?

**Method:** Start from raw WAV data. Derive every claim from measurements. Explicitly mark where evidence becomes inference. Be honest about what we don't know.

## A/B Test Setup
- Same code run on both platforms (DJ Dave kick+clap at 130 BPM)
- Desktop: recorded via Sonic Pi Rec button (DiskOut synthdef inside scsynth → reads bus 0)
- Web: recorded via Chromium Rec button (MediaRecorder → captures Web Audio output)
- **CAVEAT: These are DIFFERENT recording paths.** Desktop records inside scsynth. Web records outside, after the full Web Audio chain. We assume the Web Audio chain is unity gain (verified from code), but this assumption could be wrong.
- Two isolated tests: only_Drums (kick pattern) and no_Drums (clap + echo + reverb)

## Honest Limitations
- We have NOT tested with raw OSC commands sent directly to SuperSonic (bypassing our engine)
- We have NOT binary-compared the compiled synthdefs between npm and desktop
- We have NOT verified that the OSC commands sent by our engine exactly match desktop's
- The "2x louder" claim is about THESE WAV FILES, not necessarily about scsynth WASM vs native

In [ ]:
import wave, struct, math, os

def load_wav(path):
    """Load WAV file, return float samples and metadata."""
    with wave.open(path, 'rb') as w:
        n = w.getnframes()
        ch = w.getnchannels()
        sw = w.getsampwidth()
        rate = w.getframerate()
        raw = w.readframes(n)
    maxval = {1: 127, 2: 32767, 4: 2147483647}[sw]
    fmt_char = {1: 'b', 2: 'h', 4: 'i'}[sw]
    samples = struct.unpack(f'<{n*ch}{fmt_char}', raw)
    floats = [s / maxval for s in samples]
    mono = [sum(floats[i:i+ch])/ch for i in range(0, len(floats), ch)]
    return {'mono': mono, 'stereo': floats, 'rate': rate, 'channels': ch,
            'bits': sw * 8, 'frames': n, 'duration': n / rate, 'path': path}

def compute_stats(wav):
    """Compute audio statistics from loaded WAV."""
    stereo = wav['stereo']
    peak = max(abs(s) for s in stereo)
    rms = math.sqrt(sum(s*s for s in stereo) / len(stereo))
    clipping = sum(1 for s in stereo if abs(s) > 0.95) / len(stereo) * 100
    crest = peak / rms if rms > 0 else 0
    rate, ch = wav['rate'], wav['channels']
    per_sec = []
    for sec in range(int(wav['duration'])):
        s, e = sec * rate * ch, min((sec + 1) * rate * ch, len(stereo))
        chunk = stereo[s:e]
        if chunk: per_sec.append(math.sqrt(sum(x*x for x in chunk) / len(chunk)))
    return {'peak': peak, 'rms': rms, 'clipping_pct': clipping,
            'crest_factor': crest, 'per_sec_rms': per_sec}

def spectrogram_data(mono, rate, window_ms=20, hop_ms=10, n_bins=128):
    """Compute spectrogram: list of (time, frequency_magnitudes) using windowed DFT."""
    window_samples = int(window_ms * rate / 1000)
    hop_samples = int(hop_ms * rate / 1000)
    max_freq = rate / 2
    frames = []
    pos = 0
    while pos + window_samples <= len(mono):
        chunk = mono[pos:pos + window_samples]
        # Hann window
        N = len(chunk)
        windowed = [chunk[i] * (0.5 - 0.5 * math.cos(2 * math.pi * i / N)) for i in range(N)]
        # DFT at selected frequency bins
        mags = []
        for b in range(n_bins):
            freq = (b + 1) * max_freq / n_bins
            k = freq * N / rate
            re = sum(windowed[n] * math.cos(2 * math.pi * k * n / N) for n in range(0, N, 4)) * 4
            im = sum(windowed[n] * math.sin(2 * math.pi * k * n / N) for n in range(0, N, 4)) * 4
            mag = math.sqrt(re*re + im*im) / N
            mags.append(20 * math.log10(mag + 1e-10))  # dB
        frames.append((pos / rate, mags))
        pos += hop_samples
    return frames, max_freq, n_bins

def print_spectrogram(frames, max_freq, n_bins, title, max_time=2.0, freq_bins_to_show=32):
    """ASCII spectrogram — shows frequency content over time."""
    print(f'\n{title}')
    print(f'Y-axis: frequency (0-{max_freq:.0f} Hz, bottom=low, top=high)')
    print(f'X-axis: time (0-{max_time:.1f}s)')
    print(f'Brightness: . - + * # (quiet → loud)\n')
    
    # Select frames within time range
    sel_frames = [(t, mags) for t, mags in frames if t <= max_time]
    if not sel_frames: return
    
    # Downsample time and frequency for ASCII display
    time_step = max(1, len(sel_frames) // 80)
    freq_step = max(1, n_bins // freq_bins_to_show)
    
    # Find global min/max for normalization
    all_mags = [m for _, mags in sel_frames for m in mags]
    vmin, vmax = min(all_mags), max(all_mags)
    
    chars = ' .:-+*#@'
    for fi in range(freq_bins_to_show - 1, -1, -1):  # top = high freq
        freq_idx = fi * freq_step
        freq_hz = (freq_idx + 1) * max_freq / n_bins
        row = f'{freq_hz:6.0f}Hz |'
        for ti in range(0, len(sel_frames), time_step):
            _, mags = sel_frames[ti]
            if freq_idx < len(mags):
                val = (mags[freq_idx] - vmin) / (vmax - vmin + 1e-10)
                idx = min(len(chars) - 1, int(val * (len(chars) - 1)))
                row += chars[idx]
        print(row)
    # Time axis
    n_cols = len(range(0, len(sel_frames), time_step))
    print(f'       +{"─" * n_cols}')
    print(f'        0{" " * (n_cols // 2 - 1)}{max_time:.1f}s')

print('Utilities loaded (with spectrogram support).')

## Step 1: Load the A/B Test WAVs

**What this proves:** The files exist and are valid WAV. Nothing more.

In [ ]:
BASE = 'latest_test'
drums_desktop = load_wav(f'{BASE}/only_Drums/OriginalSonicPi_only_Drums.wav')
drums_web     = load_wav(f'{BASE}/only_Drums/SonicPiWeb_only_Drums.wav')
clap_desktop  = load_wav(f'{BASE}/no_Drums/OriginalSonicPi_no_Drums.wav')
clap_web      = load_wav(f'{BASE}/no_Drums/SonicPiWeb_no_Drums.wav')

for label, wav in [('Drums Desktop', drums_desktop), ('Drums Web', drums_web),
                    ('Clap Desktop', clap_desktop), ('Clap Web', clap_web)]:
    print(f'{label}: {wav["duration"]:.1f}s, {wav["channels"]}ch, {wav["rate"]}Hz, {wav["bits"]}bit')

## Step 2: Raw Level Comparison

**What this proves:** These specific WAV files have different RMS/peak values.

**What this does NOT prove:** That scsynth WASM is louder. The WAVs were recorded through DIFFERENT paths (DiskOut vs MediaRecorder) by DIFFERENT engines (Sonic Pi Ruby vs Sonic Pi Web JS). The difference could be in the recording, the engine, or scsynth.

In [ ]:
drums_d_stats = compute_stats(drums_desktop)
drums_w_stats = compute_stats(drums_web)
clap_d_stats  = compute_stats(clap_desktop)
clap_w_stats  = compute_stats(clap_web)

print('RAW WAV MEASUREMENTS (direct, no inference):')
print(f'{"":20} {"Desktop":>10} {"Web":>10} {"Ratio":>10}')
print('-' * 55)
for label, d, w in [('Drums RMS', drums_d_stats['rms'], drums_w_stats['rms']),
                     ('Drums Peak', drums_d_stats['peak'], drums_w_stats['peak']),
                     ('Drums Clipping%', drums_d_stats['clipping_pct'], drums_w_stats['clipping_pct']),
                     ('Drums Crest', drums_d_stats['crest_factor'], drums_w_stats['crest_factor']),
                     ('', 0, 0),
                     ('Clap RMS', clap_d_stats['rms'], clap_w_stats['rms']),
                     ('Clap Peak', clap_d_stats['peak'], clap_w_stats['peak']),
                     ('Clap Clipping%', clap_d_stats['clipping_pct'], clap_w_stats['clipping_pct']),
                     ('Clap Crest', clap_d_stats['crest_factor'], clap_w_stats['crest_factor'])]:
    if not label: print(); continue
    ratio = w / d if d > 0 else 0
    print(f'{label:20} {d:10.4f} {w:10.4f} {ratio:9.2f}x')

print(f'\nFACT: The web WAVs are louder. Drums {drums_w_stats["rms"]/drums_d_stats["rms"]:.2f}x, Clap {clap_w_stats["rms"]/clap_d_stats["rms"]:.2f}x.')
print(f'FACT: The ratios differ (2.21 vs 1.82) — not a constant factor.')
print(f'\nNOT YET PROVEN: WHERE the difference originates (engine? recording? scsynth?).')

## Step 3: Per-Second Stability

**What this proves:** The ratio is sustained, not a startup transient or recording artifact.

In [ ]:
for test_label, d_stats, w_stats in [('DRUMS', drums_d_stats, drums_w_stats),
                                       ('CLAP', clap_d_stats, clap_w_stats)]:
    d_sec, w_sec = d_stats['per_sec_rms'], w_stats['per_sec_rms']
    print(f'{test_label}:')
    ratios = []
    for i in range(min(len(d_sec), len(w_sec))):
        if d_sec[i] > 0.01 and w_sec[i] > 0.01:
            r = w_sec[i] / d_sec[i]
            ratios.append(r)
            print(f'  sec {i}: desktop={d_sec[i]:.4f}  web={w_sec[i]:.4f}  ratio={r:.2f}x')
    if ratios:
        avg = sum(ratios) / len(ratios)
        std = math.sqrt(sum((r - avg)**2 for r in ratios) / len(ratios))
        print(f'  Mean: {avg:.2f}x ± {std:.2f}')
    print()

## Step 4: Spectrogram Analysis

**What this shows:** Time-frequency content of each WAV. Allows visual comparison of:
- Are the same frequency bands active at the same times?
- Does the web version have extra frequency content (distortion from clipping)?
- Does the temporal structure match (same rhythm, same decay)?

In [ ]:
# Compute spectrograms — use 24 bins and 1.5s window for readable ASCII output
def first_hit_time(mono, rate, threshold=0.05):
    for i, s in enumerate(mono):
        if abs(s) > threshold: return i / rate
    return 0

# Drums — align to first hit, show 1.5 seconds
d_hit = first_hit_time(drums_desktop['mono'], drums_desktop['rate'], 0.1)
w_hit = first_hit_time(drums_web['mono'], drums_web['rate'], 0.1)

# Shift mono arrays to start from first hit for aligned comparison
def shift_mono(wav, hit_time):
    start = int(hit_time * wav['rate'])
    return wav['mono'][start:]

drums_d_aligned = shift_mono(drums_desktop, d_hit)
drums_w_aligned = shift_mono(drums_web, w_hit)

print('SPECTROGRAM COMPARISON — DRUMS (aligned to first kick)')
print('Each spectrogram shows 1.5s from first kick hit.\n')

sg_d, mf_d, nb_d = spectrogram_data(drums_d_aligned, drums_desktop['rate'], 
                                      window_ms=20, hop_ms=10, n_bins=64)
sg_w, mf_w, nb_w = spectrogram_data(drums_w_aligned, drums_web['rate'],
                                      window_ms=20, hop_ms=10, n_bins=64)

print_spectrogram(sg_d, mf_d, nb_d, 'DESKTOP DRUMS', max_time=1.5, freq_bins_to_show=24)
print_spectrogram(sg_w, mf_w, nb_w, 'WEB DRUMS', max_time=1.5, freq_bins_to_show=24)

In [ ]:
# Clap — same alignment
d_hit = first_hit_time(clap_desktop['mono'], clap_desktop['rate'], 0.05)
w_hit = first_hit_time(clap_web['mono'], clap_web['rate'], 0.05)

clap_d_aligned = shift_mono(clap_desktop, d_hit)
clap_w_aligned = shift_mono(clap_web, w_hit)

sg_d, mf_d, nb_d = spectrogram_data(clap_d_aligned, clap_desktop['rate'],
                                      window_ms=20, hop_ms=10, n_bins=64)
sg_w, mf_w, nb_w = spectrogram_data(clap_w_aligned, clap_web['rate'],
                                      window_ms=20, hop_ms=10, n_bins=64)

print('SPECTROGRAM COMPARISON — CLAP+FX (aligned to first hit)')
print_spectrogram(sg_d, mf_d, nb_d, 'DESKTOP CLAP', max_time=2.0, freq_bins_to_show=24)
print_spectrogram(sg_w, mf_w, nb_w, 'WEB CLAP', max_time=2.0, freq_bins_to_show=24)

## Step 5: Spectral Band Comparison

**What this shows:** Normalized frequency distribution — are the two outputs the same instrument at different levels, or spectrally different?

In [ ]:
def band_energy_simple(mono, rate, start_sec, dur_sec, lo_hz, hi_hz):
    start = int(start_sec * rate)
    end = min(start + int(dur_sec * rate), len(mono))
    chunk = mono[start:end]
    N = len(chunk)
    if N < 100: return 0
    windowed = [chunk[i] * (0.5 - 0.5 * math.cos(2 * math.pi * i / N)) for i in range(N)]
    k_lo = max(1, int(lo_hz * N / rate))
    k_hi = min(N // 2, int(hi_hz * N / rate))
    energy = 0
    step = max(1, (k_hi - k_lo) // 50)
    for k in range(k_lo, k_hi + 1, step):
        re = sum(windowed[n] * math.cos(2 * math.pi * k * n / N) for n in range(0, N, 8)) * 8
        im = sum(windowed[n] * math.sin(2 * math.pi * k * n / N) for n in range(0, N, 8)) * 8
        energy += re*re + im*im
    return math.sqrt(energy / max(1, (k_hi - k_lo) // step + 1)) / N

BANDS = {'sub_bass': (20,80), 'bass': (80,250), 'low_mid': (250,500),
         'mid': (500,2000), 'high_mid': (2000,6000), 'presence': (6000,12000),
         'brilliance': (12000,20000)}

for test_name, d_mono, w_mono, rate in [
    ('DRUMS', drums_d_aligned, drums_w_aligned, drums_desktop['rate']),
    ('CLAP+FX', clap_d_aligned, clap_w_aligned, clap_desktop['rate']),
]:
    d_bands = {name: band_energy_simple(d_mono, rate, 0, 0.3, lo, hi) for name, (lo,hi) in BANDS.items()}
    w_bands = {name: band_energy_simple(w_mono, rate, 0, 0.3, lo, hi) for name, (lo,hi) in BANDS.items()}
    d_total = sum(d_bands.values())
    w_total = sum(w_bands.values())
    print(f'\n{test_name} — Normalized spectral profile (300ms from first hit):')
    print(f'{"Band":>12}  {"Desktop%":>9}  {"Web%":>7}  {"Ratio":>7}  Note')
    for name in BANDS:
        d_pct = d_bands[name] / d_total * 100 if d_total > 0 else 0
        w_pct = w_bands[name] / w_total * 100 if w_total > 0 else 0
        ratio = w_pct / d_pct if d_pct > 0.1 else 0
        note = ''
        if ratio > 1.3: note = '<< WEB has more'
        elif 0 < ratio < 0.7: note = '<< DESKTOP has more'
        print(f'{name:>12}  {d_pct:8.1f}%  {w_pct:6.1f}%  {ratio:6.2f}x  {note}')

## Step 6: Hypothesis Testing

**Hypothesis A:** The difference is a constant gain factor applied before the mixer's non-linear processing (Limiter + clip2). The non-linearity makes the *apparent* ratio vary by signal.

**Test:** If constant factor K → loud signals (drums) should show LOWER apparent ratio (limiter compresses them more). Quiet signals (clap) should show the TRUE ratio.

In [ ]:
K_clap = clap_w_stats['rms'] / clap_d_stats['rms']
K_drums = drums_w_stats['rms'] / drums_d_stats['rms']

print(f'Clap ratio (quiet signal, less non-linear distortion): {K_clap:.2f}x')
print(f'Drums ratio (loud signal, clips at 3.4%):              {K_drums:.2f}x')
print()
print(f'Hypothesis A predicts: drums ratio < clap ratio')
print(f'  (loud signal + limiter compression → reduced apparent ratio)')
print(f'Observation: drums ratio ({K_drums:.2f}) > clap ratio ({K_clap:.2f})')
print(f'  → OPPOSITE of prediction')
print(f'\n>>> HYPOTHESIS A FAILS.')
print(f'>>> The factor is NOT a simple constant gain + non-linear compression.')
print(f'>>> The difference is genuinely signal-dependent.')
print(f'\nPossible explanations for signal-dependent ratio:')
print(f'  - Different UGens produce different level ratios in WASM vs native')
print(f'  - Our engine sends different params than desktop for some signals')
print(f'  - FX processing path differs (echo/reverb produce different levels)')
print(f'  - The recording paths capture at different points with signal-dependent behavior')

## Step 7: Evidence Chain — Honest Provenance

Every claim categorized by strength of evidence.

In [ ]:
claims = [
    ('MEASURED', 'Web WAV files are 1.8-2.2x louder than desktop WAV files',
     'Direct RMS computation on the WAV files'),
    ('MEASURED', 'The ratio is sustained over time',
     'Per-second RMS ratios stable at 2.0-2.5x'),
    ('MEASURED', 'The ratio differs between signals (2.2x drums, 1.8x clap)',
     'Two separate A/B tests with different code'),
    ('MEASURED', 'Web drums clip (3.4%), desktop drums don\'t (0%)',
     'Sample-level clipping count from WAV data'),
    ('MEASURED', 'Mixer is alive and changes output when amp is changed',
     'amp=1 test: RMS 0.074 vs amp=6: RMS 0.42 — 5.7x ratio for 6x amp'),
    ('VERIFIED', 'AudioWorklet source has no gain/scaling',
     'Read scsynth_audio_worklet.js: direct .set() copy'),
    ('VERIFIED', 'Web Audio chain is unity gain',
     'ChannelMerger spec, AnalyserNode passive, GainNode=1.0'),
    ('VERIFIED', 'Mixer params match desktop code',
     'pre_amp=0.2, amp=6 — compared against studio.rb source'),
    ('VERIFIED', 'autoConnect vs manual routing makes no difference',
     'Separate capture: RMS 0.205 vs 0.200'),
    ('', '', ''),
    ('INFERRED', 'The difference is AFTER our engine\'s parameter normalization',
     'OSC trace shows same params as desktop trace. BUT: not independently verified.'),
    ('INFERRED', 'Difference is inside scsynth WASM (by elimination)',
     'All external stages verified → must be internal. BUT: our engine code\n'
     '   was NOT bypassed. Could still be our engine sending wrong OSC.'),
    ('', '', ''),
    ('UNPROVEN', 'scsynth WASM UGens produce different output than native',
     'Plausible, would explain signal-dependent ratio. NOT tested.'),
    ('UNPROVEN', 'Desktop recording path (DiskOut) captures at same level as Web (MediaRecorder)',
     'ASSUMED to be equivalent. Not verified independently.'),
    ('UNPROVEN', 'Our engine sends identical OSC commands as desktop Sonic Pi',
     'OSC trace LOOKS right. But trace shows what WE format, not raw OSC bytes.\n'
     '   Need raw OSC isolation test to verify.'),
    ('UNPROVEN', 'Compiled synthdefs in npm are binary-identical to desktop',
     'npm README says "from Sonic Pi". Not hash-compared.'),
]

for status, claim, evidence in claims:
    if not status: print(); continue
    icon = {'MEASURED': '✅', 'VERIFIED': '🔍', 'INFERRED': '🔶', 'UNPROVEN': '❓'}[status]
    print(f'{icon} [{status}] {claim}')
    print(f'   Evidence: {evidence}')
    print()

## Step 8: What We Actually Know

An honest summary separating fact from inference.

In [ ]:
print('=' * 70)
print('HONEST CONCLUSIONS')
print('=' * 70)
print()
print('FACT: These web WAV files are 1.8-2.2x louder than these desktop WAV files.')
print('FACT: The ratio is signal-dependent, not a constant.')
print('FACT: The mixer is processing audio (amp test proves it).')
print('FACT: The AudioWorklet and Web Audio chain have no gain (source verified).')
print()
print('INFERENCE (not proven):')
print('  The difference is inside scsynth WASM — by process of elimination.')
print('  But we have NOT eliminated our engine code from the signal path.')
print()
print('THE MISSING TEST:')
print('  Send raw OSC commands directly to SuperSonic, bypassing our entire')
print('  engine (SoundLayer, AudioInterpreter, SonicPiEngine). Record the output.')
print('  Compare against desktop.')
print()
print('  If still 2x louder → confirmed SuperSonic/scsynth WASM issue')
print('  If matches desktop → our engine is the cause (sends wrong params/extra synths)')
print()
print('  Until this test is run, we cannot attribute the difference to SuperSonic.')